In [ ]:
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
from torchvision import datasets
from PIL import Image
import bidsio
import pickle
import sys
sys.path.append('../')
from helpers import *
from fns_ingp import *
from bandlimited_signal import *


device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
seed = 0 
BIDS_LOADER = bidsio.BIDSLoader(data_entities=[{'subject': '',
                                              'session': '',
                                              'suffix': 'T1w',
                                              'space': 'MNI152NLin2009aSym'}],
                              target_entities=[],
                              data_derivatives_names=['ATLAS'],
                              batch_size=1,
                              root_dir='./atlas/data/test/')


In [ ]:
def load_gt(dataset, id_val, bandlimit = 0.6):
    if dataset=='div2k':
        image_pil = Image.open(f'.s/DIV2K/DIV2K_train_LR_x8/0{str(id_val)}x8.png')
        RES = 128
    elif dataset == 'sphere':
        image = SparseSphereSignal(dimension=2, length=128, bandlimit=bandlimit, seed=id_val, generate=False).signal
    elif dataset == 'bandlimited':
        image = BandlimitedSignal(dimension=2, length=128, bandlimit=bandlimit, seed=id_val, generate=False).signal
    elif dataset == 'mri':
        RES = 128
        tmp = BIDS_LOADER.load_sample(idx = id_val, data_only=True) / 255.0
        vol = resize(tmp, (1, RES, RES, RES))[0]
        slice_idx = 48
        image = vol[:, :, slice_idx]  # (RES, RES)
    else:
        raise ValueError(f"Dataset {dataset} not supported.")

    if dataset not in ['sphere', 'bandlimited', 'mri']:  
        image = np.array(image_pil.convert('L').resize((RES, RES))) / 255.0 
        
    return image

def compute_d(p, w):
    return 2 * p * w + 2 * w + 1

def compute_params(model_size, model_params, param_to_vary):
    if param_to_vary == 'mapping_size':
        return int(np.floor((model_size - 2 * model_params['width'] - 1) / (2 * model_params['width'])))
    else:
        return int(np.floor((model_size - 1) / (2 * model_params['mapping_size'] + 1)))


In [ ]:
dataset = 'mri'
if dataset in ['sphere', 'bandlimited']:
    image_idx = 1234
else:
    image_idx = 131
image = load_gt(dataset, image_idx)

learning_rate, iters = 5e-4, 10000
mask = None
if dataset == 'mri':
    mask = np.fft.fftshift(np.ones((128, 128))).astype(np.complex64)

# varying hash_table_size
model_params = {
    'num_levels': 5,
    'hash_table_size': 0,
    'base_resolution': 16,
    'n_features_per_level': 2.0,
}
param_vals = [5, 9]

# varying num_levels
# model_params = {
#     'num_levels': 0,
#     'hash_table_size': 11,
#     'base_resolution': 16,
#     'n_features_per_level': 2.0,
# }
# param_vals = [2, 3]

if model_params['num_levels'] == 0:
    param_to_vary = 'num_levels'
else:    
    param_to_vary = 'hash_table_size'
outputs = {}
to_save_outputs = {}
for param_val in param_vals:
    model_params[param_to_vary] = param_val
    print(f'{param_to_vary}: {model_params[param_to_vary]}')
    model_config = (int(model_params['num_levels']), int(model_params['hash_table_size']), model_params['base_resolution'], model_params['n_features_per_level'])
    output = fit_instant_ngp(image, model_config, iters=iters, learning_rate=learning_rate, log_interval = 1000, seed=seed, device=device, count_params=True)
    outputs[f'{model_params[param_to_vary]}'] = output     
    to_save_outputs[f'{model_params[param_to_vary]}'] = output['best_pred']
    error = np.linalg.norm(image.flatten() - output['best_pred'].flatten())                       
    print(f"Error: {error:.3e}, Loss: {output['best_loss']:.3e}")

with open(f"2d_{dataset}/ingp.pkl", "wb") as f:
    pickle.dump(to_save_outputs, f)

In [ ]:
plot_error_heatmaps(image, outputs, model_name="Instant-NGP")